# 06 — Conditional model strategy bake-off

This notebook compares six alternative SRKW-versus-Transient strategies against the guarded marine-transport baseline: regularized logistic regression, a spline/GAM-like model, group-robust histogram boosting, a learned water-graph diffusion stack, group-DRO, and an evidence-regime mixture of experts. A calibrated constrained ensemble is trained with separate inner selection and calibration partitions.

The main estimate uses five encounter-held-out folds. Rolling-time, held-out-H3-region, and held-out-source results are residual-model stress tests over frozen OOF base scores—not end-to-end deployment estimates. Nothing in this notebook updates production artifacts.

In [ ]:
import os
from pathlib import Path
import json
import pandas as pd

from experiment_support import resolve_release_paths
from strategy_bakeoff_experiment import run_strategy_bakeoff_experiment

paths = resolve_release_paths()
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'strategy_bakeoff'
paths.release_id, paths.snapshot_id

In [ ]:
result = run_strategy_bakeoff_experiment(output_dir, paths)
manifest = json.loads(result['manifest_path'].read_text())
manifest['recommended_research_challenger'], manifest['promotion_eligible']

## Overall comparisons

Metrics use source-by-observed-class post-stratification weights. The balanced 10,000-row OOF sample is therefore adjusted toward the labeled encounter reference, but it still cannot identify the unknown-label distribution.

In [ ]:
comparison = result['comparison']
display(comparison.sort_values(['scheme', 'log_loss'])[[
    'scheme', 'model', 'n', 'brier', 'log_loss', 'roc_auc',
    'equal_mass_ece_10', 'calibration_intercept', 'calibration_slope'
]].reset_index(drop=True))

## Robustness gates

A challenger must improve primary Brier and log loss, meet calibration tolerances, avoid more than 10% Brier degradation in every sufficiently large source/era/region/evidence stratum, and avoid more than 10% overall degradation in every residual stress scheme.

In [ ]:
display(result['research_decisions'].sort_values('model').reset_index(drop=True))
display(result['gate_summary'].sort_values(['model', 'stratification']).reset_index(drop=True))
display(result['stress_gates'].sort_values(['scheme', 'model']).reset_index(drop=True))

## Ensemble composition and interpretation

The ensemble is diagnostic unless it also survives a full end-to-end rolling/spatial/source refit. Large gains from flexible spatial or group models can be source or era shortcuts even though `SOURCE` itself is excluded as a biological feature.

In [ ]:
display(result['ensemble_diagnostics'])
print('Research recommendation:', result['recommendation'])
print('Promotion blockers:')
for blocker in manifest['promotion_blockers']:
    print('-', blocker)